# PyToch: Workflow

In [ ]:
import torch
import numpy as np
from pathlib import Path
from torch import nn
from torch import optim
import matplotlib.pyplot as plt
from common import plot_predictions

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
print(f'PyTorch: version {torch.__version__}')
print(f'PyTorch: {device.upper()} mode')

## Linear Regression Model

In [ ]:
a = 0.7
b = 0.3

X = torch.arange(start=0, end=1, step=0.02).unsqueeze(1)
Y = a*X + b
X.shape,Y.shape

In [ ]:
split = int(len(X) * 0.8)
x_train, y_train = X[:split], Y[:split]
x_test, y_test = X[split:], Y[split:]
len(x_train), len(y_train), len(x_test), len(y_test)

In [ ]:
plot_predictions(x_train, y_train, x_test, y_test)

In [ ]:
# Define model
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(
            1,
            requires_grad=True,
            dtype=torch.float
        ))
        self.bias = nn.Parameter(torch.randn(
            1,
            requires_grad=True,
            dtype=torch.float
        ))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weights * x + self.bias

In [ ]:
class LayeredLinearRegressionModel(nn.Module):
    def __init__(self, in_features: int = 1, out_features: int = 1, bias: bool = True):
        super().__init__()
        self.linear1 = nn.Linear(in_features, out_features, bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear1(x)

In [ ]:
# Create a model from our definition
model0 = LayeredLinearRegressionModel()
# Send to device
model0.to(device)
# Check model device
next(model0.parameters()).device

### Training Loop

0. Loop through training data
1. Forward pass (`forward()` method of the model is involved) to make predictions
2. Calculate the loss (compare predictions with ground truth labels)
3. Optimize zero grad
4. Backward pass to calculate the gradients of each of the parameters w.r.t. the loss
5. Adjust model's parameters in order to reduce the loss (**gradient descent**)

In [ ]:
# Setup a loss function
loss_fn = nn.L1Loss()
# Setup an optimizer
optimizer = optim.SGD(model0.parameters(),
                      lr=1e-3,
                      momentum=0.9)

In [ ]:
epochs = 300

epoch_count = []
tr_loss_values = []
ts_loss_values = []

x_train = x_train.to(device)
y_train = y_train.to(device)
x_test = x_test.to(device)
y_test = y_test.to(device)

for epoch in range(epochs):
    ### Training
    # 0. Enable training mode (sets all parameters to require gradients where `requires_grad=True`)
    model0.train(mode=True)
    # 1. Forward pass
    y_pred = model0(x_train)
    # 2. Calculate the loss
    loss = loss_fn(y_pred, y_train)
    # 3. Optimize zero grad
    optimizer.zero_grad()
    # 4. Backward pass
    loss.backward()
    # 5. Adjust model's parameters
    optimizer.step()

    ### Testing
    # 0. Disable training mode (turn off certain modules)
    model0.train(mode=False)
    with torch.inference_mode():
        # 1. Forward pass
        test_pred = model0(x_test)
        # 2. Calculate the loss
        test_loss = loss_fn(test_pred, y_test)

    ### Gather statistic
    if epoch % 10 == 0:
        epoch_count.append(epoch)
        tr_loss_values.append(loss)
        ts_loss_values.append(test_loss)
        print(f'Epoch: {epoch}, Loss: {loss}, Test loss: {test_loss}')

In [ ]:
# Using context manager to get 'initial' predictions
with torch.inference_mode():
    y_pred_old = model0(x_test.to(device))

plot_predictions(x_train.cpu().numpy(),
                 y_train.cpu().numpy(),
                 x_test.cpu().numpy(),
                 y_test.cpu().numpy(),
                 predictions=y_pred_old.cpu())

In [ ]:
state = model0.state_dict()
print(f'Named parameters:')
print(f'   learned: {state['linear1.weight']}, target: {a}')
print(f'   learned: {state['linear1.bias']}, target: {b}')

In [ ]:
# Plot loss and test loss values for each epoch
y1_values = np.array(torch.tensor(tr_loss_values, device='cpu').numpy())
y2_values = np.array(torch.tensor(ts_loss_values, device='cpu').numpy())
plt.plot(epoch_count, y1_values, label='Train Loss')
plt.plot(epoch_count, y2_values, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and test loss curves')
_ = plt.legend()

## Saving and Loading a Model State

1. `torch.save()` - allows to save a PyTorch object in Python's pickle format
2. `torch.load()` - allows to load a PyTorch object from Python's pickle format
3. `torch.nn.Module.load_state_dict()` - allows to load a model's saved state dict

In [ ]:
# 1. Prepare models directory
MODEL_PATH = Path('files')/'pt_workflow'
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create a model save path
MODEL_NAME = 'pt_workflow.pt'
torch.save(model0.state_dict(), MODEL_PATH/MODEL_NAME)

In [ ]:
# Plot predictions

model0.train(mode=False)
with torch.inference_mode():
    y_pred_new = model0(x_test)

plot_predictions(x_train.cpu().numpy(),
                 y_train.cpu().numpy(),
                 x_test.cpu().numpy(),
                 y_test.cpu().numpy(),
                 predictions=y_pred_old.cpu())

In [ ]:
model1 = LayeredLinearRegressionModel()
# Load state dict from the file
model1.load_state_dict(torch.load(MODEL_PATH/MODEL_NAME))
# Send to device
model1.to(device)
# Check model device
next(model1.parameters()).device

## Saving and Loading a Model

The disadvantage of this approach is that the serialized data is bound to the specific classes and the exact directory structure used when the model is saved. The reason for this is because pickle does not save the model class itself. Rather, it saves a path to the file containing the class, which is used during load time. Because of this, your code can break in various ways when used in other projects or after refactors.

In [ ]:
# Plot predictions
model1.train(mode=False)
with torch.inference_mode():
    y_pred_new = model1(x_test)

plot_predictions(x_train.cpu().numpy(),
                 y_train.cpu().numpy(),
                 x_test.cpu().numpy(),
                 y_test.cpu().numpy(),
                 predictions=y_pred_old.cpu())

In [ ]:
torch.save(model0, MODEL_PATH/MODEL_NAME)

In [ ]:
model2 = torch.load(MODEL_PATH/MODEL_NAME, weights_only=False)
model2.to(device)

In [ ]:
# Plot predictions
model2.train(mode=False)
with torch.inference_mode():
    y_pred_new = model2(x_test)

plot_predictions(x_train.cpu().numpy(),
                 y_train.cpu().numpy(),
                 x_test.cpu().numpy(),
                 y_test.cpu().numpy(),
                 predictions=y_pred_old.cpu())